# Gold or Foreign Currency? Protecting Savings from Inflation in Poland (2013–2026)

Jakub Wojciechowski 49872 - Mini Project - Network and Business Analytics

In [1]:
!pip install eurostat plotly nbformat scipy

# 1. Problem and objective

**Problem:** Polish households and small firms keep their savings in PLN. Inflation and a weaker złoty make this money worth less over time. Where should they keep savings to protect their value – in PLN cash, in foreign currencies (EUR, USD, CHF, GBP) or in gold?

**Objective:** collect daily gold prices and exchange rates from the National Bank of Poland (NBP) and monthly inflation from Eurostat for 2013–2026, and explore the data (EDA).

**Approach:** theory first – I started with the problem and then collected the data needed to answer it.

# 2. Data collection

| Source | Data | Frequency |
|---|---|---|
| NBP API | gold price (PLN per 1 gram) | daily |
| NBP API | average exchange rates of EUR, USD, CHF, GBP (table A) | daily |
| Eurostat | HICP price index for Poland (2015 = 100) | monthly |

The NBP API gives max. one year of data per request, so I download the data year by year in a loop. Gold prices in the API start in 2013.

In [2]:
import requests
import pandas as pd
import time

START_YEAR = 2013
END_YEAR = 2026
END_DATE = "2026-10-16"  # last day with data

## 2.1 Gold price (NBP)

In [3]:
all_gold = []

for year in range(START_YEAR, END_YEAR + 1):
    if year == END_YEAR:
        end = END_DATE
    else:
        end = f"{year}-12-31"

    url = f"https://api.nbp.pl/api/cenyzlota/{year}-01-01/{end}/?format=json"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        all_gold.extend(data)
        print(f"Gold {year}: {len(data)} days")
    else:
        print(f"Failed to fetch gold {year}, status code: {response.status_code}")

    time.sleep(1)

Gold 2013: 251 days
Gold 2014: 252 days
Gold 2015: 254 days
Gold 2016: 252 days
Gold 2017: 251 days
Gold 2018: 251 days
Gold 2019: 251 days
Gold 2020: 255 days
Gold 2021: 254 days
Gold 2022: 252 days
Gold 2023: 251 days
Gold 2024: 252 days
Gold 2025: 251 days
Gold 2026: 202 days


In [4]:
df_gold = pd.DataFrame(all_gold)
df_gold = df_gold.rename(columns={'data': 'Date', 'cena': 'Gold'})
df_gold.to_csv('nbp_gold_price.csv', index=False)
df_gold.head()

,Date,Gold
0,2013-01-02,165.83
1,2013-01-03,166.97
2,2013-01-04,167.43
3,2013-01-07,167.98
4,2013-01-08,167.26


## 2.2 Exchange rates (NBP)

In [5]:
currencies = ['EUR', 'USD', 'CHF', 'GBP']
all_rates = []

for code in currencies:
    for year in range(START_YEAR, END_YEAR + 1):
        if year == END_YEAR:
            end = END_DATE
        else:
            end = f"{year}-12-31"

        url = f"https://api.nbp.pl/api/exchangerates/rates/A/{code}/{year}-01-01/{end}/?format=json"
        response = requests.get(url)

        if response.status_code == 200:
            data = response.json()
            rates = data.get("rates", [])
            all_rates.extend([{
                "Date": rate['effectiveDate'],
                "Currency": code,
                "Rate": rate['mid']
            } for rate in rates])
            print(f"{code} {year}: {len(rates)} days")
        else:
            print(f"Failed to fetch {code} {year}, status code: {response.status_code}")

        time.sleep(1)

EUR 2013: 251 days
EUR 2014: 252 days
EUR 2015: 254 days
EUR 2016: 252 days
EUR 2017: 251 days
EUR 2018: 252 days
EUR 2019: 251 days
EUR 2020: 255 days
EUR 2021: 254 days
EUR 2022: 252 days
EUR 2023: 251 days
EUR 2024: 252 days
EUR 2025: 251 days
EUR 2026: 202 days
USD 2013: 251 days
USD 2014: 252 days
USD 2015: 254 days
USD 2016: 252 days
USD 2017: 251 days
USD 2018: 252 days
USD 2019: 251 days
USD 2020: 255 days
USD 2021: 254 days
USD 2022: 252 days
USD 2023: 251 days
USD 2024: 252 days
USD 2025: 251 days
USD 2026: 202 days
CHF 2013: 251 days
CHF 2014: 252 days
CHF 2015: 254 days
CHF 2016: 252 days
CHF 2017: 251 days
CHF 2018: 252 days
CHF 2019: 251 days
CHF 2020: 255 days
CHF 2021: 254 days
CHF 2022: 252 days
CHF 2023: 251 days
CHF 2024: 252 days
CHF 2025: 251 days
CHF 2026: 202 days
GBP 2013: 251 days
GBP 2014: 252 days
GBP 2015: 254 days
GBP 2016: 252 days
GBP 2017: 251 days
GBP 2018: 252 days
GBP 2019: 251 days
GBP 2020: 255 days
GBP 2021: 254 days
GBP 2022: 252 days
GBP 2023: 25

In [6]:
# long format (Date, Currency, Rate) – panel data
df_fx_long = pd.DataFrame(all_rates)
df_fx_long.to_csv('nbp_exchange_rates_panel.csv', index=False)

# wide format – one column per currency
df_fx = df_fx_long.pivot_table(index='Date', columns='Currency', values='Rate')
df_fx.head()

Currency,CHF,EUR,GBP,USD
Date,,,,
2013-01-02,3.3647,4.0671,5.0026,3.0660
2013-01-03,3.3717,4.0770,5.0296,3.1005
2013-01-04,3.4106,4.1248,5.0873,3.1700
2013-01-07,3.4094,4.1218,5.0767,3.1618
2013-01-08,3.4123,4.1263,5.0605,3.1457


## 2.3 Inflation (Eurostat)

In 2026 Eurostat moved to a new product classification. The old table `prc_hicp_midx` stops in December 2025, so I use the new table `prc_hicp_minr`. I download the full table (about 100,000 rows – all countries and product groups) and keep only the total index for Poland.

In [7]:
import eurostat

df_cpi = eurostat.get_data_df('prc_hicp_minr')
df_cpi = df_cpi[(df_cpi['geo\\TIME_PERIOD'] == 'PL') & (df_cpi['coicop18'] == 'TOTAL') & (df_cpi['unit'] == 'I15')]
df_cpi.head()

,freq,unit,coicop18,geo\TIME_PERIOD,1996-01,1996-02,1996-03,1996-04,1996-05,1996-06,...,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
19769,M,I15,TOTAL,PL,42.96,43.62,44.28,45.22,45.78,46.18,...,154.72,155.86,156.45,157.88,158.76,158.53,158.1,158.78,159.2,160.24


In [8]:
df_cpi = df_cpi.rename(columns={'geo\\TIME_PERIOD': 'geo'})
df_cpi = df_cpi.melt(id_vars=['freq', 'unit', 'coicop18', 'geo'],
                     var_name='Month', value_name='HICP').dropna(subset=['HICP'])
df_cpi = df_cpi[df_cpi['Month'] >= '2013-01']
df_cpi = df_cpi[['Month', 'HICP']]
df_cpi.to_csv('eurostat_hicp_poland.csv', index=False)
df_cpi.tail()

,Month,HICP
364,2026-05,158.53
365,2026-06,158.10
366,2026-07,158.78
367,2026-08,159.20
368,2026-09,160.24


## 2.4 One daily table

In [9]:
df_gold = df_gold.set_index('Date')

# gold and currencies side by side, matched by date
df = pd.concat([df_gold, df_fx], axis=1)
df = df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')
df = df.reset_index(drop=True)
df.to_csv('gold_fx_daily.csv', index=False)

print(df.shape)
df.head()

(3480, 6)


,Date,Gold,CHF,EUR,GBP,USD
0,2013-01-02,165.83,3.3647,4.0671,5.0026,3.0660
1,2013-01-03,166.97,3.3717,4.0770,5.0296,3.1005
2,2013-01-04,167.43,3.4106,4.1248,5.0873,3.1700
3,2013-01-07,167.98,3.4094,4.1218,5.0767,3.1618
4,2013-01-08,167.26,3.4123,4.1263,5.0605,3.1457


# 3. Exploratory data analysis (EDA)

## 3.1 Overview

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3480 entries, 0 to 3479
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    3480 non-null   datetime64[ns]
 1   Gold    3479 non-null   float64       
 2   CHF     3480 non-null   float64       
 3   EUR     3480 non-null   float64       
 4   GBP     3480 non-null   float64       
 5   USD     3480 non-null   float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 163.3 KB


In [11]:
print(df['Date'].min(), df['Date'].max())
print(df_cpi['Month'].min(), df_cpi['Month'].max())

2013-01-02 00:00:00 2026-10-16 00:00:00
2013-01 2026-09


The daily table has **3,480 days** (2 Jan 2013 – 16 Oct 2026) and 6 columns: the date and 5 prices in PLN, all numeric. Inflation is monthly and covers **165 months** (Jan 2013 – Sep 2026), so it ends earlier than the daily data.

## 3.2 Data issues

In [12]:
df.isna().sum()

Date    0
Gold    1
CHF     0
EUR     0
GBP     0
USD     0
dtype: int64

In [13]:
df[df['Gold'].isna()]

,Date,Gold,CHF,EUR,GBP,USD
1478,2018-11-12,NaN,3.7809,4.2962,4.9027,3.8163


**One gold price is missing** – 12 Nov 2018. It was an extra public holiday in Poland (100 years of independence), but NBP still published exchange rates that day. All other values are complete.

In [14]:
df.duplicated().sum()

np.int64(0)

In [15]:
# number of days between two quotes
gaps = df['Date'].diff()
gaps.value_counts()

Date
1 days    2704
3 days     670
2 days      52
4 days      49
5 days       3
6 days       1
Name: count, dtype: int64

**No duplicates.** Most quotes are 1 day apart. 3-day gaps are weekends and 2-, 4-, 5- and 6-day gaps are public holidays (e.g. Easter, Christmas). This is normal for market data – there are no quotes on these days, so these are not missing values.

## 3.3 Handling the issues

- **Missing gold price (1 day):** I use the price from the day before.
- **Weekends and holidays:** no quotes on these days, so I leave them as they are.
- **Daily vs monthly data:** I take monthly averages of prices to match them with monthly inflation.
- **Different end dates:** inflation ends in September 2026, so October 2026 is dropped from the inflation part.

In [16]:
df['Gold'] = df['Gold'].ffill()
df.isna().sum()

Date    0
Gold    0
CHF     0
EUR     0
GBP     0
USD     0
dtype: int64

In [17]:
assets = ['Gold', 'EUR', 'USD', 'CHF', 'GBP']

# monthly averages + inflation in one table
monthly = df.groupby(df['Date'].dt.to_period('M'))[assets].mean()
monthly.index = monthly.index.astype(str)

df_month = pd.concat([monthly, df_cpi.set_index('Month')], axis=1).dropna()
df_month.to_csv('gold_fx_hicp_monthly.csv', index_label='Month')
print(df_month.shape)
df_month.tail()

(165, 6)


,Gold,EUR,USD,CHF,GBP,HICP
2026-05,536.073000,4.241750,3.632880,4.636525,4.899375,158.53
2026-06,505.195238,4.258324,3.698767,4.625786,4.929476,158.10
2026-07,494.196087,4.318057,3.779717,4.666474,5.056722,158.78
2026-08,524.164286,4.310786,3.719114,4.604186,5.035190,159.20
2026-09,525.429545,4.344609,3.774159,4.606595,5.064141,160.24


After matching prices with inflation there are **165 months**. October 2026 was dropped because there is no inflation data for it yet.

## 3.4 Descriptive statistics

### Prices (PLN)

In [18]:
df[assets].describe()

,Gold,EUR,USD,CHF,GBP
count,3480.000000,3480.000000,3480.000000,3480.000000,3480.000000
mean,228.457152,4.345163,3.795956,4.105087,5.135120
std,110.256346,0.167286,0.365219,0.437384,0.296673
min,116.350000,3.982200,3.004200,3.306700,4.585200
25%,147.277500,4.230350,3.646250,3.792975,4.916900
50%,190.020000,4.294150,3.803800,4.073550,5.055550
75%,263.175000,4.446025,3.983575,4.538550,5.314275
max,624.920000,4.964700,5.038100,5.120000,6.076900


In [19]:
df[assets].median()

Gold    190.02000
EUR       4.29415
USD       3.80380
CHF       4.07355
GBP       5.05555
dtype: float64

In [20]:
# total change 2013–2026 (%)
(df[assets].iloc[-1] / df[assets].iloc[0] - 1) * 100

Gold    213.483688
EUR       7.351676
USD      26.637312
CHF      39.979790
GBP       3.438212
dtype: float64

**Gold changed the most:** from 166 to 520 PLN per gram (**+213%**). Currencies changed much less: CHF +40%, USD +27%, EUR +7%, GBP +3%. EUR is the most stable – it stayed between 3.98 and 4.96 PLN. The mean gold price (228) is much higher than the median (190), because gold was much more expensive in the last years.

### Daily returns (%)

Prices have very different levels (gold 116–625 PLN, EUR 4–5 PLN), so I compare daily log returns in %.

In [21]:
import numpy as np

for col in assets:
    df[col + '_ret'] = 100 * np.log(df[col]).diff()

ret_cols = ['Gold_ret', 'EUR_ret', 'USD_ret', 'CHF_ret', 'GBP_ret']
df[ret_cols].describe()

,Gold_ret,EUR_ret,USD_ret,CHF_ret,GBP_ret
count,3479.000000,3479.000000,3479.000000,3479.000000,3479.000000
mean,0.032842,0.002039,0.006788,0.009667,0.000972
std,1.097085,0.343184,0.623102,0.552112,0.533989
min,-9.711656,-1.799146,-4.196450,-2.603810,-3.671648
25%,-0.539570,-0.187448,-0.365298,-0.266471,-0.295045
50%,0.025622,-0.004660,-0.005164,-0.014440,0.002000
75%,0.597459,0.170101,0.346805,0.255530,0.293007
max,7.690901,2.620291,4.043750,15.287779,3.048816


In [22]:
df[ret_cols].median()

Gold_ret    0.025622
EUR_ret    -0.004660
USD_ret    -0.005164
CHF_ret    -0.014440
GBP_ret     0.002000
dtype: float64

In [23]:
df[ret_cols].skew()

Gold_ret   -0.328378
EUR_ret     0.568521
USD_ret     0.327060
CHF_ret     6.481904
GBP_ret     0.013252
dtype: float64

In [24]:
df[ret_cols].kurtosis()

Gold_ret      6.906175
EUR_ret       5.441746
USD_ret       3.296445
CHF_ret     170.526660
GBP_ret       3.364769
dtype: float64

Most returns are almost symmetric (skewness close to 0). **CHF is the exception** (skewness 6.5, kurtosis 170) – this comes from one extreme day (see below). Kurtosis is above 0 for all assets (pandas shows excess kurtosis, normal = 0), so very big daily moves happen more often than in a normal distribution.

In [25]:
# yearly volatility (252 trading days)
df[ret_cols].std() * np.sqrt(252)

Gold_ret    17.415682
EUR_ret      5.447885
USD_ret      9.891441
CHF_ret      8.764500
GBP_ret      8.476814
dtype: float64

**Gold is the most volatile asset – 17.4% per year**, 3 times more than EUR (5.4%). USD (9.9%), CHF (8.8%) and GBP (8.5%) are in the middle. Average daily returns are close to 0 for all assets.

In [26]:
# biggest daily rise and fall
for col in ret_cols:
    print(df.sort_values(col, ascending=False).head(1)[['Date', col]])
    print(df.sort_values(col).head(1)[['Date', col]])

          Date  Gold_ret
878 2016-06-27  7.690901
         Date  Gold_ret
73 2013-04-16 -9.711656
           Date   EUR_ret
2316 2022-03-07  2.620291
          Date   EUR_ret
992 2016-12-07 -1.799146
           Date  USD_ret
2316 2022-03-07  4.04375
           Date  USD_ret
2490 2022-11-14 -4.19645
          Date    CHF_ret
511 2015-01-15  15.287779
          Date  CHF_ret
519 2015-01-27 -2.60381
          Date   GBP_ret
975 2016-11-14  3.048816
          Date   GBP_ret
878 2016-06-27 -3.671648


The biggest moves match known events:
- **CHF +15.3% on 15 Jan 2015** – the Swiss central bank stopped holding the franc at 1.20 per euro.
- Gold −9.7% on 16 Apr 2013 – the big gold price crash in April 2013.
- 27 Jun 2016 – days after the Brexit vote (gold +7.7%, GBP −3.7%).
- 7 Mar 2022 – after Russia attacked Ukraine the złoty got much weaker (EUR +2.6%, USD +4.0%).

## 3.5 Visualization

In [27]:
import plotly.graph_objects as go

fig = go.Figure()

for col in assets:
    fig.add_trace(go.Scatter(
        x=df['Date'],
        y=df[col] / df[col].iloc[0] * 100,
        mode='lines',
        name=col
    ))

fig.update_layout(
    title='Value of 100 PLN put in each asset on 2 Jan 2013',
    xaxis_title='Date',
    yaxis_title='Value (PLN)',
    hovermode='x unified',
    template='plotly_white'
)

fig.show()

Gold grew the most, especially from 2024 – at the peak (**3 Mar 2026**) 100 PLN became 377 PLN. But it also had long falls: in 2013 it lost about 30% (100 → 70 PLN at the start of 2014). Currencies moved in a much narrower range (about 92–164).

In [28]:
# real value = value after inflation
inflation = df_month['HICP'] / df_month['HICP'].iloc[0]

fig = go.Figure()

for col in assets:
    fig.add_trace(go.Scatter(
        x=df_month.index,
        y=df_month[col] / df_month[col].iloc[0] / inflation * 100,
        mode='lines',
        name=col
    ))

fig.add_trace(go.Scatter(
    x=df_month.index,
    y=100 / inflation,
    mode='lines',
    name='Cash (PLN)',
    line=dict(color='black', dash='dash')
))

fig.update_layout(
    title='Real value of 100 PLN after inflation (2013 prices)',
    xaxis_title='Month',
    yaxis_title='Real value (PLN)',
    hovermode='x unified',
    template='plotly_white'
)

fig.show()

In [29]:
# real value in the last month
print('Cash (PLN):', 100 / inflation.iloc[-1])
df_month[assets].iloc[-1] / df_month[assets].iloc[0] / inflation.iloc[-1] * 100

Cash (PLN): 62.51248127808287


Gold    196.305375
EUR      65.611080
USD      75.761520
CHF      85.487390
GBP      63.647453
dtype: float64

**This is the key result.** Prices in Poland went up by 60%, so **100 PLN kept in cash is worth only 62.5 PLN** today. All currencies also lost real value: CHF 85, USD 76, EUR 66, GBP 64. **Only gold protected the savings – 196 PLN**, almost twice the starting value. But not all the time: from 2013 until mid-2019 gold was below 100 (lost real value) almost every month.

In [30]:
import plotly.express as px

for col in ret_cols:
    px.histogram(df, x=col, nbins=100, title=f'Distribution of {col}').show()

Most daily returns are small and close to 0, but a few days have very big moves. These big days happen more frequently than in a normal distribution – it is called fat tails. Gold has the widest histogram, so its daily moves are the biggest.

In [31]:
px.box(df, y=ret_cols, title='Box Plot of daily returns (%)')

All assets have many outliers. The biggest one is CHF +15% (Jan 2015). I keep all outliers – they are real market events, not data errors.

In [32]:
px.imshow(df[ret_cols].corr().round(2), text_auto=True, title='Correlation Matrix of daily returns')

**Gold is not correlated with currencies** (−0.10 to 0.02). Currencies move together (0.45–0.68) – when the złoty gets weaker, it gets weaker against all of them. So gold can diversify savings, while holding several currencies gives less diversification.

In [33]:
fig = go.Figure()

for col in ret_cols:
    fig.add_trace(go.Scatter(
        x=df['Date'],
        y=df[col].rolling(window=252).std() * np.sqrt(252),
        mode='lines',
        name=col
    ))

fig.update_layout(
    title='1-year rolling volatility (%)',
    xaxis_title='Date',
    yaxis_title='Volatility (%)',
    hovermode='x unified',
    template='plotly_white'
)

fig.show()

Volatility changes over time. CHF was highest in 2015 (17.6%), GBP in 2016–2017 (Brexit), USD and EUR in 2022 (war in Ukraine). **Gold volatility is now the highest in the whole period (about 26%)**, compared to about 10% in 2018.

## 3.6 Normality of returns

In [34]:
from scipy.stats import shapiro

for col in ret_cols:
    stat, p = shapiro(df[col].dropna())
    print(f"{col}: Shapiro-Wilk = {stat:.3f}, p-value = {p:.4f}")

Gold_ret: Shapiro-Wilk = 0.944, p-value = 0.0000
EUR_ret: Shapiro-Wilk = 0.943, p-value = 0.0000
USD_ret: Shapiro-Wilk = 0.971, p-value = 0.0000
CHF_ret: Shapiro-Wilk = 0.790, p-value = 0.0000
GBP_ret: Shapiro-Wilk = 0.968, p-value = 0.0000


The p-value is below 0.05 for all assets, so **none of the returns is normally distributed**. It matches the fat tails above – very big daily moves happen more often than a normal distribution expects. So the real risk of these assets is higher than it looks from the standard deviation alone.

# 4. Overall findings

- **Data:** 3,480 days of gold and 4 exchange rates from NBP and 165 months of inflation from Eurostat (2013–2026). The data is clean: 1 missing gold price, no duplicates, gaps only on weekends and holidays.
- **Inflation:** prices in Poland went up by 60%, so 100 PLN in cash lost about 37% of its value.
- **Currencies did not protect savings:** all of them lost real value (CHF best with 85 PLN, GBP worst with 64 PLN).
- **Gold was the only asset that protected savings** (196 PLN in real terms), but with the highest risk: 17.4% yearly volatility, a 30% fall in 2013 and real losses until mid-2019.
- **Gold is not correlated with currencies**, so it can diversify savings. Returns are not normal (fat tails).
- **Next steps:** predict future gold prices (for example ARIMA), compare gold with keeping money in a bank deposit and check if mixing gold and currencies lowers the risk.